In [79]:
import pandas as pd
import os

# Scratch file for interactively visualizing *parallel-ml-bench* data

DATA_ROOT = 'data'
TEST_FILE = 'test_parallel_ml_bench:260726-001603.processed.jsonl'

def load_df(fname):
    df = pd.read_json(os.path.join(DATA_ROOT, fname), lines=True)
    df.set_index('bench')
    return df

df = load_df(TEST_FILE)
df


,tag,bench,args,config,cwd,exp,trials,procs,cmd,host,timestamp,elapsed,returncode,binary_bytes,binary_md5,warmup_result_secs,test_results_secs
0,primes,primes,-N 100000000,mlton,mpl,time,1,1,/usr/bin/time -v bin/primes.mlton.bin -N 10000...,big-mpl,2026-07-26 00:16:07.322971,37.394843,0,263352,a3b644e74ee4bb51fe83899fabe50754,"[1.5629, 1.5746, 1.5366, 1.5772]","[1.5655999999999999, 1.5841, 1.6031, 1.6045, 1..."
1,primes,primes,-N 100000000,mlton-baseline,mpl,time,1,1,/usr/bin/time -v bin/primes.mlton-baseline.bin...,big-mpl,2026-07-26 00:16:44.720352,38.567872,0,266904,398fd0d5c0177427e49a6dd8479fe6b3,"[1.6278000000000001, 1.6412, 1.624099999999999...","[1.6026, 1.6038999999999999, 1.613300000000000..."


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from dataclasses import dataclass

# Use system latex
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"]
})

CHARTS_ROOT = 'charts/'

CHECKSUM_FIELD = 'binary_md5'
COMPILER_NAME_FIELD = 'config'

# Plots data from the parallel-ml-bench suite
def plot(df, values='runTime', title='', out_filename='', abbrevs=('MLton0', 'MLton1')):
    # Remove columns we don't care about
    filtered = df[['bench', COMPILER_NAME_FIELD, values, CHECKSUM_FIELD]]
    # Remove benchmarks witn identical binaries
    filtered = df[filtered.groupby('bench')[CHECKSUM_FIELD].transform('nunique') > 1]
    pivot = filtered.pivot(index='bench', columns=COMPILER_NAME_FIELD, values=values)
    # Calculate the ratio (<1 is good, >1 is bad)
    abs_ratio = pivot[abbrevs[1]] / pivot[abbrevs[0]]
    # Convert to relative_pct (<0% is good, >0% is bad)
    pivot['relative_pct'] = (abs_ratio - 1) * 100
    ax = (pivot['relative_pct']).plot(kind='bar')

    # Calculate the absolute geomean (1.0 is neutral)
    abs_geomean = np.exp(np.mean(np.log(abs_ratio.dropna())))
    # Scale to match the metric for relative_pct
    geomean_pct = (abs_geomean - 1) * 100
    # Add a text box
    textstr = f'Geomean: {geomean_pct:+.1f}\\%'
    props = dict(boxstyle='square,pad=0.5', facecolor='white', alpha=0.9, edgecolor='black', linewidth=0.5)
    ax.text(0.95, 0.95, textstr, transform=ax.transAxes, fontsize=10,
            verticalalignment='top', horizontalalignment='right', bbox=props)
    
    ax.set_title(title)
    ax.set_xlabel('Benchmark name')
    ax.set_ylabel(r'Relative \% $\frac{\mathrm{test}}{\mathrm{base}} - 1 \times 100\%$')
    ax.yaxis.set_major_formatter(ticker.PercentFormatter())
    path = os.path.join(CHARTS_ROOT, f'{out_filename}.pdf')
    print(f'NOT saving chart to {path}')
    #print(f'Saving chart to {path}')
    #plt.savefig(path, format='pdf', bbox_inches='tight')
    plt.show()



In [81]:
@dataclass(frozen=True)
class PlotConfig:
    values_column: str
    title: str
    out_filename: str

def plot_mlton_vs_mlton(data, type_name, flavor_name):
    print(f'Plotting file for {type_name} flattening (MLton vs MLton): {data}')
    df = load_df(data)
    configs = [
        PlotConfig(values_column='test_results_secs',
                   title='Run time comparison', 
                   out_filename=f'{type_name}_{flavor_name}_run_mlton_vs_mlton'),
      #  PlotConfig(values_column='compileTime',
      #             title='Compile time comparison',
      #             out_filename=f'{type_name}_mlton_compile_mlton_vs_mlton'),
      #  PlotConfig(values_column='binarySize',
      #             title='Binary size comparison',
      #             out_filename=f'{type_name}_mlton_size_mlton_vs_mlton'),
    ]
    for c in configs:
        plot_mlton(df, values=c.values_column, title=c.title, out_filename=c.out_filename)


# Generate all parallel-ml-bench MLton-vs-MLton charts
def plot_test_mlton_vs_mlton():
    plot_mlton_vs_mlton(TEST_FILE, 'test')


plot_test_mlton_vs_mlton()

Plotting file for test flattening (MLton vs MLton): test_parallel_ml_bench:260726-001603.processed.jsonl


NameError: name 'flavor_name' is not defined

In [ ]:
plot_mlton_tuple_flattening()

In [ ]:
plot_mlton_con_flattening()